# Notebook 02b — Génération Q&R ASYNCHRONE (3–4 datasets)

Version optimisée avec **Groq** + asyncio : requêtes concurrentes (attention au rate limit).
Meme logique que 02_dataset_builder.ipynb mais avec asyncio pour la vitesse.

**Outputs** : `train.json` (1200 paires) + `test.json` (300 paires) — aligné sur le notebook 02 (Option B ; juridique optionnel)

## 0. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation

In [ ]:
# groq + nest_asyncio (Colab) + json-repair
!pip install -q groq nest_asyncio json-repair

## 2. Imports, configuration et clé API

In [ ]:
import os, json, time, asyncio, random, getpass, re
import nest_asyncio
from groq import Groq
from tqdm.notebook import tqdm

# Indispensable pour utiliser 'await' dans les cellules Colab
nest_asyncio.apply()

RAW_PATH       = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

GROQ_MODEL      = 'llama-3.1-8b-instant'  # aligné notebook 03
BATCH_SIZE      = 3    # Groq — réduire si 429 (free tier)
THROTTLE_S      = 0.5  # pause légère entre batches
MAX_CONTENT     = 4000

print(f'Modèle      : {GROQ_MODEL}')
print(f'Batch size  : {BATCH_SIZE} requêtes concurrentes')
print(f'nest_asyncio activé.')

In [ ]:
# Saisie sécurisée de la clé — https://console.groq.com → API Keys (préfixe gsk_)
api_key = getpass.getpass('Entre ta clé Groq API : ').strip()
if not api_key.startswith('gsk_'):
    raise ValueError('Clé Groq invalide (doit commencer par gsk_)')
groq_client = Groq(api_key=api_key)
print(f'Client Groq initialisé. Modèle : {GROQ_MODEL}')

## 3. Chargement des données brutes

In [ ]:
def load_json(path, label=''):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé ({label}) : {len(data)} docs")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Introuvable : {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des datasets bruts...")
wiki_docs    = load_json(os.path.join(RAW_PATH, 'wikipedia_technique.json'), 'Wikipedia technique')
hal_docs     = load_json(os.path.join(RAW_PATH, 'hal.json'),                 'HAL multisauts FR')
lemonde_docs = load_json(os.path.join(RAW_PATH, 'lemonde.json'),             'Le Monde temporel')
legal_docs   = load_json(os.path.join(RAW_PATH, 'code_route.json'),          'Code de la route (PDF)')
HAS_LEGAL    = len(legal_docs) > 0
print(f"\nTotal : {len(wiki_docs)+len(hal_docs)+len(lemonde_docs)+len(legal_docs)} documents (juridique : {len(legal_docs)})")

def infer_recency(date_str):
    try:
        year = int(str(date_str)[:4])
        if year >= 2024: return 'récent'
        if year >= 2022: return 'intermédiaire'
        return 'fondamental'
    except Exception:
        return 'inconnu'

for doc in wiki_docs + hal_docs + lemonde_docs + legal_docs:
    doc['recency_category'] = infer_recency(doc.get('date', ''))
print('  recency_category calculé depuis les dates des documents.')

## 4. Génération async des paires Q&R

In [ ]:
# ── Prompt standard (Wikipedia + Le Monde) ───────────────────────────────────
STANDARD_PROMPT = """Tu es un expert. À partir du texte ci-dessous, génère EXACTEMENT 5 paires question-réponse EN FRANÇAIS.

Génère dans cet ordre :
1. Question FACTUELLE (fait précis, date, chiffre, entité nommée)
2. Question FACTUELLE (idem)
3. Question de SYNTHÈSE (compare ou résume plusieurs concepts)
4. Question de SYNTHÈSE (idem)
5. Question de COMPRÉHENSION (causalité, implication, pourquoi/comment)

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du texte (20-150 mots) qui contient/justifie la réponse.
INTERDIT d'écrire "Texte source", "référence au texte" ou le titre seul.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"..."}}
Types autorisés : "factuel", "synthese", "comprehension"

Exemple :
[{{"question":"En quelle année X a été fondé ?","answer":"2020","context":"X a été fondé en 2020 par des chercheurs issus de Google Brain, avec pour objectif de...","type":"factuel"}}]

Texte source :
{content}"""

# ── Prompt Arxiv simples (3 questions par papier) ────────────────────────────
HAL_SIMPLE_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique EN FRANÇAIS, génère EXACTEMENT 3 questions factuelles simples EN FRANÇAIS.

Chaque question porte sur UN SEUL fait (méthode utilisée, métrique obtenue, dataset employé, contribution principale).
Chaque réponse tient en 1-2 phrases courtes.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du résumé (20-100 mots) qui contient la réponse.
INTERDIT d'écrire "Texte source" ou similaire.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"simple"}}

Exemple :
[{{"question":"Quel dataset est utilisé pour l'évaluation ?","answer":"MMLU","context":"We evaluate our model on the MMLU benchmark, achieving state-of-the-art performance across 57 tasks.","type":"simple"}}]

Résumé :
{content}"""

# ── Prompt Arxiv complexes / multi-sauts (2 questions par papier) ─────────────
HAL_COMPLEX_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique EN FRANÇAIS, génère EXACTEMENT 2 questions complexes EN FRANÇAIS.

Ces questions nécessitent de RELIER PLUSIEURS INFORMATIONS du texte pour répondre (multi-sauts de raisonnement).
Par exemple : "Pourquoi la méthode X obtient-elle de meilleurs résultats que Y sur Z ?"
Chaque réponse fait 3-4 phrases et synthétise plusieurs éléments du résumé.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT 1-2 extraits du résumé (30-200 mots) qui, ensemble, permettent de répondre.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"complexe"}}

Résumé :
{content}"""

# GROQ_MODEL défini dans la cellule imports
THROTTLE_S  = 0.5    # Groq Developer — pas de rate limit pratique
MAX_CONTENT = 4000   # chars (aligné notebook 02)

In [ ]:
VALID_TYPES = {"factuel", "synthese", "comprehension", "simple", "complexe"}

import re as _re
from json_repair import repair_json

def _extract_and_repair(raw_text):
    """Extrait le JSON d'une réponse LLM et le répare si nécessaire (aligné notebook 02)."""
    match = _re.search(r'(\[.*\])', raw_text, _re.DOTALL)
    if match:
        candidate = match.group(1)
    else:
        candidate = raw_text.strip()
    return repair_json(candidate, return_objects=False)

def _sync_groq_chat(prompt):
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4096,
        temperature=0.2,
    )
    return (resp.choices[0].message.content or "").strip()

async def _call_groq_async(prompt, retries=4):
    for attempt in range(retries):
        try:
            return await asyncio.to_thread(_sync_groq_chat, prompt)
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = 5 * (2 ** attempt)
                print(f"    [QUOTA] Attente {wait}s...")
                await asyncio.sleep(wait)
            else:
                print(f"    [ERROR] {err[:80]}")
                await asyncio.sleep(2)
    return ""

def _parse_pairs(raw_text, document, forced_type=None):
    content = document.get('content', '')
    json_str = _extract_and_repair(raw_text)
    pairs = json.loads(json_str)
    if not isinstance(pairs, list):
        return []
    validated = []
    for pair in pairs:
        if not (isinstance(pair, dict) and pair.get('question') and pair.get('answer')):
            continue
        q_type = forced_type or str(pair.get('type', 'factuel')).lower().strip()
        if q_type not in VALID_TYPES:
            q_type = forced_type or 'factuel'
        ctx = str(pair.get('context', '')).strip()
        if len(ctx) < 30:
            ctx = content[:400]
        validated.append({
            "question":      str(pair['question']).strip(),
            "answer":        str(pair['answer']).strip(),
            "context":       ctx,
            "source_id":     document.get('id', ''),
            "source":        document.get('source', ''),
            "langue":        "fr",
            "title":         document.get('title', ''),
            "date":          document.get('date', ''),
            "dataset_type":      document.get('dataset_type', ''),
            "question_type":     q_type,
            "recency_category": document.get('recency_category', 'inconnu'),
        })
    return validated

async def generate_standard_qa_async(document, retries=4):
    content = document.get('content', '')
    if len(content) < 100:
        return []
    prompt = STANDARD_PROMPT.format(content=content[:MAX_CONTENT])
    for attempt in range(retries):
        try:
            raw = await _call_groq_async(prompt)
            if not raw:
                continue
            pairs = _parse_pairs(raw, document)
            if pairs:
                return pairs
        except (json.JSONDecodeError, ValueError):
            await asyncio.sleep(2 ** attempt)
    return []

async def generate_arxiv_qa_async(document, retries=4):
    content = document.get('content', '')
    if len(content) < 100:
        return []
    all_pairs = []
    for prompt_template, forced_type, n_expected in [
        (HAL_SIMPLE_PROMPT,  'simple',   3),
        (HAL_COMPLEX_PROMPT, 'complexe', 2),
    ]:
        prompt = prompt_template.format(content=content[:MAX_CONTENT])
        for attempt in range(retries):
            try:
                raw = await _call_groq_async(prompt)
                if not raw:
                    break
                pairs = _parse_pairs(raw, document, forced_type=forced_type)
                if pairs and len(pairs) >= n_expected:
                    all_pairs.extend(pairs[:n_expected])
                    break
            except (json.JSONDecodeError, ValueError):
                await asyncio.sleep(2 ** attempt)
    return all_pairs

# Aligné 02_dataset_builder : relances par doc jusqu'à 5 paires valides
MIN_PAIRS_PER_DOC = 5
MAX_DOC_ROUNDS = 18

async def _generate_standard_until_quota_async(doc, label=""):
    """Accumule des paires à questions distinctes sur plusieurs tours (évite rester bloqué à 1/5)."""
    seen, best = set(), []
    for r in range(MAX_DOC_ROUNDS):
        batch = await generate_standard_qa_async(doc)
        for p in batch:
            k = str(p.get('question', '')).strip().lower()[:160]
            if not k or k in seen:
                continue
            seen.add(k)
            best.append(p)
            if len(best) >= MIN_PAIRS_PER_DOC:
                return best[:MIN_PAIRS_PER_DOC]
        await asyncio.sleep(THROTTLE_S + min(r, 6))
    if len(best) < MIN_PAIRS_PER_DOC:
        print(f"  [WARN] {label} '{str(doc.get('title',''))[:50]}' : {len(best)}/{MIN_PAIRS_PER_DOC} paires")
    return best[:MIN_PAIRS_PER_DOC] if len(best) >= MIN_PAIRS_PER_DOC else best

async def _generate_hal_until_quota_async(doc):
    """Accumule 3 simples + 2 complexes (questions distinctes par type)."""
    seen_s, seen_c = set(), set()
    simple, complexe = [], []
    for r in range(MAX_DOC_ROUNDS):
        batch = await generate_arxiv_qa_async(doc)
        for p in batch:
            k = str(p.get('question', '')).strip().lower()[:160]
            if not k:
                continue
            qt = str(p.get('question_type', 'simple')).lower()
            if qt == 'complexe':
                if k in seen_c:
                    continue
                seen_c.add(k)
                complexe.append(p)
            else:
                if k in seen_s:
                    continue
                seen_s.add(k)
                simple.append(p)
        if len(simple) >= 3 and len(complexe) >= 2:
            return simple[:3] + complexe[:2]
        await asyncio.sleep(THROTTLE_S + min(r, 6))
    merged = simple[:3] + complexe[:2]
    if len(merged) < MIN_PAIRS_PER_DOC:
        print(f"  [WARN] HAL '{str(doc.get('title',''))[:50]}' : {len(merged)}/{MIN_PAIRS_PER_DOC} paires")
    return merged[:MIN_PAIRS_PER_DOC] if len(merged) >= MIN_PAIRS_PER_DOC else merged

print("Fonctions async chargees.")

In [ ]:
async def generate_all_qa(docs_wiki, docs_hal, docs_lemonde, docs_legal=None):
    """Génère toutes les Q&R en parallèle par batches (juridique = prompt standard)."""
    sem = asyncio.Semaphore(BATCH_SIZE)

    async def process_doc(doc):
        async with sem:
            dtype = doc.get('dataset_type', '')
            if dtype == 'multisauts':
                pairs = await _generate_hal_until_quota_async(doc)
            else:
                lbl = {"technique": "Wiki", "temporel": "News", "juridique": "CDR"}.get(dtype, "")
                pairs = await _generate_standard_until_quota_async(doc, lbl)
            return pairs

    all_docs = docs_wiki + docs_hal + docs_lemonde
    if docs_legal:
        all_docs = all_docs + list(docs_legal)
    tasks = [process_doc(d) for d in all_docs]
    results = []
    failed  = 0
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Q&R async"):
        try:
            pairs = await coro
            results.extend(pairs)
        except Exception as e:
            print(f"  [ERROR] {e}")
            failed += 1
        await asyncio.sleep(THROTTLE_S)
    print(f"\nGeneres : {len(results)} paires  |  echecs : {failed}")
    return results

# Run
nest_asyncio.apply()
import random
legal_sample = []
if HAS_LEGAL:
    if len(legal_docs) < 20:
        raise RuntimeError(
            "code_route.json : au moins 20 segments pour 100 paires juridiques. Voir 02_dataset_builder / 01_scraping."
        )
    random.seed(42)
    _s = legal_docs[:]
    random.shuffle(_s)
    legal_sample = _s[:20]

all_qa_pairs = asyncio.run(generate_all_qa(wiki_docs, hal_docs, lemonde_docs, legal_sample or None))

def _quota_check():
    """Cible = nb de documents bruts × 5 (ex. 72 articles Wiki → 360 paires, pas 500)."""
    m = MIN_PAIRS_PER_DOC
    tw = sum(1 for p in all_qa_pairs if p.get('dataset_type') == 'technique')
    th = sum(1 for p in all_qa_pairs if p.get('dataset_type') == 'multisauts')
    tl = sum(1 for p in all_qa_pairs if p.get('dataset_type') == 'temporel')
    tj = sum(1 for p in all_qa_pairs if p.get('dataset_type') == 'juridique')
    rows = [
        ("Wikipedia", tw, len(wiki_docs) * m),
        ("HAL", th, len(hal_docs) * m),
        ("Le Monde", tl, len(lemonde_docs) * m),
    ]
    if HAS_LEGAL:
        rows.append(("Code route", tj, len(legal_sample) * m))
    for name, got, tgt in rows:
        if got < tgt:
            raise RuntimeError(f"Quota insuffisant — {name}: {got}/{tgt} paires.")
    print("  [OK] Quotas atteints (cibles = len(docs)×5).")

_quota_check()

## 5. Mélange et division train / test

Split **stratifié** sur `recency_category` (proportions train/test alignées sur le pool de chaque source, seed 42).

In [ ]:
import random
import math
random.seed(42)

def exact_split(pairs, n_train, n_test, label='', stratify_key='recency_category'):
    """Sélectionne n_train + n_test ; si pool < demandé, split proportionnel (HAL complexe max 200)."""
    total0 = len(pairs)
    if total0 == 0:
        return [], []
    pool = pairs[:]
    random.shuffle(pool)
    T_req = n_train + n_test
    if total0 > T_req:
        pool = pool[:T_req]
    total = len(pool)
    n_train_o, n_test_o = n_train, n_test
    T = n_train + n_test
    if total < T:
        print(f"  [WARN] {label}: pool={total} < {n_train_o}+{n_test_o} → split proportionnel.")
        ratio = (n_train / T) if T > 0 else 0.5
        n_train = max(0, min(total, int(round(total * ratio))))
        n_test = total - n_train
        if n_test_o > 0 and n_test == 0 and total > 1:
            n_test = min(n_test_o, max(1, total // 5))
            n_train = total - n_test
        T = n_train + n_test
    if not stratify_key:
        return pool[:n_train], pool[n_train:T]
    cats = sorted({p.get(stratify_key, 'inconnu') for p in pool})
    if len(cats) <= 1:
        return pool[:n_train], pool[n_train:T]
    by_cat = {c: [p for p in pool if p.get(stratify_key, 'inconnu') == c] for c in cats}
    for c in cats:
        random.shuffle(by_cat[c])
    sizes = {c: len(by_cat[c]) for c in cats}
    tot = sum(sizes.values())
    floor_t = {c: int(math.floor(sizes[c] * n_train / tot)) for c in cats}
    rem = n_train - sum(floor_t.values())
    order = sorted(cats, key=lambda c: (sizes[c] * n_train / tot - floor_t[c]), reverse=True)
    alloc = dict(floor_t)
    for i in range(rem):
        alloc[order[i % len(order)]] += 1
    train, test = [], []
    for c in cats:
        lst = by_cat[c]
        k = min(alloc.get(c, 0), len(lst))
        train.extend(lst[:k])
        test.extend(lst[k:])
    random.shuffle(train)
    random.shuffle(test)
    return train, test

# ── Option B : 1200 train + 300 test ────────────────────────────────────────
# La génération async produit une seule liste `all_qa_pairs` — on segmente comme en 02 séquentiel.
if 'all_qa_pairs' not in globals():
    raise RuntimeError("Exécute d'abord la cellule « Q&R async » qui définit all_qa_pairs.")
wiki_qa = [p for p in all_qa_pairs if p.get('dataset_type') == 'technique']
lemonde_qa = [p for p in all_qa_pairs if p.get('dataset_type') == 'temporel']
legal_qa = [p for p in all_qa_pairs if p.get('dataset_type') == 'juridique']
hal_simple  = [p for p in all_qa_pairs if p.get('dataset_type') == 'multisauts' and p.get('question_type') == 'simple']
hal_complex = [p for p in all_qa_pairs if p.get('dataset_type') == 'multisauts' and p.get('question_type') == 'complexe']
print(f"Pools avant split : wiki {len(wiki_qa)} | HAL s/c {len(hal_simple)}/{len(hal_complex)} | news {len(lemonde_qa)} | juridique {len(legal_qa)}")

if HAS_LEGAL:
    wiki_train, wiki_test = exact_split(wiki_qa, 320, 80, 'Wikipedia')
    legal_train, legal_test = exact_split(legal_qa, 80, 20, 'Code route')
else:
    wiki_train, wiki_test = exact_split(wiki_qa, 400, 100, 'Wikipedia')
    legal_train, legal_test = [], []

# ── HAL : équilibre simple/complexe ──────────────────────────────────────────
hal_s_train, hal_s_test = exact_split(hal_simple,  200, 50, 'HAL simple')
hal_c_train, hal_c_test = exact_split(hal_complex, 200, 50, 'HAL complexe')
hal_train = hal_s_train + hal_c_train
hal_test  = hal_s_test  + hal_c_test
print(f"  [CHECK] HAL test : {len(hal_s_test)} simple + {len(hal_c_test)} complexe | Wiki test (technique) : {sum(1 for p in wiki_test if p.get('dataset_type')=='technique')}")

# ── Le Monde : 400 train + 100 test ──────────────────────────────────────────
lemonde_train, lemonde_test = exact_split(lemonde_qa, 400, 100, 'Le Monde')

# ── Assemblage final ─────────────────────────────────────────────────────────
train_data = wiki_train + hal_train + lemonde_train + legal_train
test_data  = wiki_test  + hal_test  + lemonde_test + legal_test
random.shuffle(train_data)
random.shuffle(test_data)

# Re-indexation des pair_id
for i, p in enumerate(train_data): p['pair_id'] = f"train_{i:04d}"
for i, p in enumerate(test_data):  p['pair_id'] = f"test_{i:04d}"

import collections
print("\n" + "=" * 55)
print("RÉPARTITION FINALE DU DATASET")
print("=" * 55)
for split_name, split in [("TRAIN (1200)", train_data), ("TEST  (300)", test_data)]:
    print(f"\n  {split_name}")
    by_ds = collections.Counter(p['dataset_type']   for p in split)
    by_qt = collections.Counter(p['question_type']  for p in split)
    by_rc = collections.Counter(p.get('recency_category', 'inconnu') for p in split)
    print(f"    Par source     : {dict(by_ds)}")
    print(f"    Par type Q     : {dict(by_qt)}")
    print(f"    Par temporalité: {dict(by_rc)}")
print(f"\n  Totaux : {len(train_data)} train + {len(test_data)} test = {len(train_data)+len(test_data)} paires")

## 6. Sauvegarde sur Drive

In [ ]:
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_path = os.path.join(PROCESSED_PATH, 'train.json')
test_path  = os.path.join(PROCESSED_PATH, 'test.json')

for data, path, label in [(train_data, train_path, 'train'), (test_data, test_path, 'test')]:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"{label}.json : {len(data)} paires  ({os.path.getsize(path)/1024:.1f} Ko)  → {path}")

## 7. Statistiques du dataset

In [ ]:
import statistics

def field_stats(pairs, field):
    counts = [len(str(p.get(field, '')).split()) for p in pairs]
    return min(counts), round(statistics.mean(counts),1), int(statistics.median(counts)), max(counts), sum(counts)

def print_stats(split_name, pairs):
    print(f'\n  ── {split_name} ({len(pairs)} paires) ──')
    print(f'  {"Champ":<12} {"Min":>5} {"Moy":>7} {"Méd":>6} {"Max":>5} {"Total":>10}')
    print(f'  {"─"*12} {"─"*5} {"─"*7} {"─"*6} {"─"*5} {"─"*10}')
    for field, label in [("question","Question"),("answer","Réponse"),("context","Contexte")]:
        mn,mv,md,mx,tot = field_stats(pairs, field)
        print(f'  {label:<12} {mn:>5} {mv:>7} {md:>6} {mx:>5} {tot:>10,}')
    sources = {}
    for p in pairs:
        s = p.get('source','?'); sources[s] = sources.get(s,0)+1
    print(f'  Sources : {dict(sorted(sources.items(), key=lambda x:-x[1]))}')

print('=' * 55)
print('STATISTIQUES DU DATASET (langue : français)')
print('=' * 55)
print_stats('TRAIN', train_data)
print_stats('TEST ', test_data)
all_text = ' '.join(p.get('question','') + ' ' + p.get('answer','') for p in all_qa_pairs)
import re as _re
vocab = set(_re.sub(r'[^a-zàâéèêëîïôùûüç\s]','',all_text.lower()).split())
print(f'\n  Vocabulaire unique : {len(vocab):,} tokens')
print(f'  Total paires       : {len(train_data)+len(test_data)} (train {len(train_data)} + test {len(test_data)})')
print('=' * 55)
print('\n✔ Notebook 02b terminé. Lancez 03_baseline_rag.ipynb pour la suite.')